In [ ]:
# ============================================
# Advanced Penetration Testing Toolkit 
# ============================================

import socket
import requests
import time
import hashlib
import os
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

lock = Lock()

# -----------------------------------
# SERVICE DETECTION
# -----------------------------------
def service_detection(port):
    services = {
        21: "FTP", 22: "SSH", 23: "Telnet", 25: "SMTP",
        53: "DNS", 80: "HTTP", 110: "POP3",
        143: "IMAP", 443: "HTTPS",
        3306: "MySQL", 8080: "HTTP-Alt"
    }
    return services.get(port, "Unknown")

# -----------------------------------
# LOGGING FUNCTION
# -----------------------------------
def log_result(data):
    with open("results.txt", "a") as f:
        f.write(data + "\n")

# -----------------------------------
# FULL PORT SCANNER (1–65535)
# -----------------------------------
def scan_port(target, port, open_ports):
    try:
        s = socket.socket()
        s.settimeout(0.3)

        result = s.connect_ex((target, port))

        if result == 0:
            service = service_detection(port)
            output = f"[OPEN] Port {port} ({service})"

            with lock:
                print(output)
                log_result(output)
                open_ports.append(port)

        s.close()
    except:
        pass

def full_port_scanner():
    target = input("Enter target IP: ")

    print("\n[+] Scanning ALL ports (1–65535)...\n")

    open_ports = []

    with ThreadPoolExecutor(max_workers=100) as executor:
        for port in range(1, 65536):
            executor.submit(scan_port, target, port, open_ports)

    print("\n========== SUMMARY ==========")

    if open_ports:
        print(f"Total Open Ports: {len(open_ports)}")
        print("Ports:", sorted(open_ports))
    else:
        print("No open ports found")

# -----------------------------------
# BANNER GRABBING
# -----------------------------------
def banner_grabber():
    target = input("Enter target IP: ")
    port = int(input("Enter port: "))

    try:
        s = socket.socket()
        s.settimeout(2)
        s.connect((target, port))
        banner = s.recv(1024)
        print("[+] Banner:", banner.decode(errors="ignore"))
    except:
        print("[-] No banner found")

# -----------------------------------
# HTTP SCANNER
# -----------------------------------
def http_scanner():
    url = input("Enter URL: ")
    try:
        r = requests.get(url, timeout=5)
        print("Status:", r.status_code)
        print("Headers:", r.headers)
    except:
        print("[-] Request failed")

# -----------------------------------
# DIRECTORY SCANNER
# -----------------------------------
def dir_scanner():
    url = input("Enter URL: ")
    paths = ["admin", "login", "dashboard"]

    for path in paths:
        full_url = url + "/" + path
        try:
            r = requests.get(full_url, timeout=3)
            if r.status_code == 200:
                print("[FOUND]", full_url)
        except:
            pass

# -----------------------------------
# SUBDOMAIN FINDER
# -----------------------------------
def subdomain_finder():
    domain = input("Enter domain: ")
    subs = ["www", "mail", "ftp", "admin"]

    for sub in subs:
        url = f"http://{sub}.{domain}"
        try:
            requests.get(url, timeout=3)
            print("[FOUND]", url)
        except:
            pass

# -----------------------------------
# RESPONSE TIME ANALYZER
# -----------------------------------
def response_time():
    url = input("Enter URL: ")
    try:
        start = time.time()
        requests.get(url)
        end = time.time()
        print(f"Response Time: {end - start:.2f}s")
    except:
        print("[-] Failed")

# -----------------------------------
# SECURITY HEADERS CHECK
# -----------------------------------
def security_check():
    url = input("Enter URL: ")
    try:
        r = requests.get(url)

        missing = []

        if "X-Frame-Options" not in r.headers:
            missing.append("X-Frame-Options")
        if "Content-Security-Policy" not in r.headers:
            missing.append("CSP")
        if "Strict-Transport-Security" not in r.headers:
            missing.append("HSTS")

        if missing:
            print("[!] Missing:", missing)
        else:
            print("[+] All headers present")

    except:
        print("[-] Failed")

# -----------------------------------
# PASSWORD POLICY CHECK
# -----------------------------------
def password_policy_check():
    pwd = input("Enter password: ")

    score = 0

    if len(pwd) >= 12:
        score += 1
    if any(c.isupper() for c in pwd):
        score += 1
    if any(c.isdigit() for c in pwd):
        score += 1
    if any(c in "!@#$%^&*" for c in pwd):
        score += 1

    if score == 4:
        print("Strong Password")
    elif score >= 2:
        print("Medium Password")
    else:
        print("Weak Password")

# -----------------------------------
# CONTROLLED BRUTE FORCE (SAFE)
# -----------------------------------
def brute_force_lab():
    print("\n[+] Controlled Brute Force Simulation\n")

    correct_password = "admin123"
    wordlist = ["1234", "password", "admin", "admin123", "test"]

    attempts = 0

    for pwd in wordlist:
        attempts += 1
        print(f"Trying: {pwd}")
        time.sleep(0.5)

        if pwd == correct_password:
            print(f"\n[+] Password FOUND: {pwd}")
            print(f"Attempts: {attempts}")
            return

    print("[-] Password not found")

# -----------------------------------
# DNS LOOKUP
# -----------------------------------
def dns_lookup():
    domain = input("Enter domain: ")
    try:
        print(socket.gethostbyname(domain))
    except:
        print("Failed")

# -----------------------------------
# HASH GENERATOR
# -----------------------------------
def hash_generator():
    text = input("Enter text: ")
    print(hashlib.sha256(text.encode()).hexdigest())

# -----------------------------------
# PING TOOL
# -----------------------------------
def ping_check():
    target = input("Enter IP or domain: ")

    print(f"\n[+] Pinging {target}...\n")

    if os.name == "nt":
        os.system(f"ping {target}")
    else:
        os.system(f"ping -c 4 {target}")

# -----------------------------------
# MAIN MENU
# -----------------------------------
def main():
    print("\n===== Advanced Toolkit =====")

    print("1. Full Port Scanner")
    print("2. Banner Grabbing")
    print("3. HTTP Scanner")
    print("4. Directory Scanner")
    print("5. Subdomain Finder")
    print("6. Response Time")
    print("7. Security Check")
    print("8. Password Checker")
    print("9. DNS Lookup")
    print("10. Hash Generator")
    print("11. Ping Tool")
    print("12. Brute Force (Lab)")
    print("13. Exit")

    choice = input("Select option: ")

    if choice == "1":
        full_port_scanner()
    elif choice == "2":
        banner_grabber()
    elif choice == "3":
        http_scanner()
    elif choice == "4":
        dir_scanner()
    elif choice == "5":
        subdomain_finder()
    elif choice == "6":
        response_time()
    elif choice == "7":
        security_check()
    elif choice == "8":
        password_policy_check()
    elif choice == "9":
        dns_lookup()
    elif choice == "10":
        hash_generator()
    elif choice == "11":
        ping_check()
    elif choice == "12":
        brute_force_lab()
    else:
        print("Exit")

# -----------------------------------
# RUN
# -----------------------------------
if __name__ == "__main__":
    print(" Use only on authorized systems\n")
    main()

 Use only on authorized systems


===== Advanced Toolkit =====
1. Full Port Scanner
2. Banner Grabbing
3. HTTP Scanner
4. Directory Scanner
5. Subdomain Finder
6. Response Time
7. Security Check
8. Password Checker
9. DNS Lookup
10. Hash Generator
11. Ping Tool
12. Brute Force (Lab)
13. Exit
